
# RAG Evaluation Notebook (EM / BLEU / METEOR)

Evaluate 5 systems:

1. `embed + finetune + rerank`
2. `embed + no_finetune + rerank`
3. `embed + finetune + no_rerank`
4. `no_embed + no_rerank + finetune`
5. `no_embed + no_rerank + no_finetune`

Pipeline:
- Load `jd_embeddings.npz` + metadata
- Build FAISS index
- Retrieval top-k (for embed systems)
- Optional rerank (`AITeamVN/Vietnamese_Reranker`)
- Generate answers with Qwen3-4B (base or LoRA fine-tuned)
- Compute EM, BLEU, METEOR


In [1]:

# Colab deps (run once), then Runtime -> Restart runtime
!pip -q uninstall -y torchao
!pip install -q --upgrade pip
!pip install -q "transformers==4.53.3" "peft==0.19.1" "accelerate>=0.33.0" sentencepiece faiss-cpu nltk sacrebleu pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.4 MB/s eta 0:00:00


In [2]:

import os
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
import faiss

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from peft import PeftModel

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# ===== Colab/Local path config =====
IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/RAG'  # change if needed
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path(DRIVE_PROJECT_ROOT)
else:
    ROOT = Path('d:/nlp')

EMB_PATH = ROOT / 'jd_embeddings.npz'
TEST_PATH = ROOT / 'data/test.jsonl'
FT_ADAPTER_PATH = ROOT / 'qwen3-4b-qlora-jd/best'
HF_CACHE = str(ROOT / 'hf_cache')
os.environ['HF_HOME'] = HF_CACHE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print('IN_COLAB =', IN_COLAB)
print('ROOT =', ROOT)
print('DEVICE =', DEVICE)


Mounted at /content/drive
IN_COLAB = True
ROOT = /content/drive/MyDrive/RAG
DEVICE = cuda


In [3]:

# 1) Load embeddings + metadata
npz = np.load(EMB_PATH, allow_pickle=True)
print('Keys:', npz.files)

embed_key = None
for k in npz.files:
    arr = npz[k]
    if isinstance(arr, np.ndarray) and arr.ndim == 2:
        embed_key = k
        break

if embed_key is None:
    raise ValueError('Kh?ng t?m th?y ma tr?n embedding 2 chi?u trong jd_embeddings.npz')

embeddings = np.asarray(npz[embed_key], dtype=np.float32)
print('Embedding key:', embed_key, '| shape =', embeddings.shape)

jd_ids = npz['jd_ids'] if 'jd_ids' in npz.files else np.array([str(i) for i in range(len(embeddings))])
texts = npz['texts'] if 'texts' in npz.files else np.array([''] * len(embeddings))
companies = npz['companies'] if 'companies' in npz.files else np.array([''] * len(embeddings))
titles = npz['titles'] if 'titles' in npz.files else np.array([''] * len(embeddings))
cities = npz['cities'] if 'cities' in npz.files else np.array([''] * len(embeddings))

N = len(embeddings)
assert all(len(x) == N for x in [jd_ids, texts, companies, titles, cities])

metadata = []
for i in range(N):
    metadata.append({
        'idx': i,
        'jd_id': str(jd_ids[i]),
        'title': str(titles[i]),
        'company': str(companies[i]),
        'city': str(cities[i]),
        'full_text': str(texts[i]),
    })

pd.DataFrame(metadata[:3])


Keys: ['embeddings', 'jd_ids', 'texts', 'companies', 'titles', 'cities']
Embedding key: embeddings | shape = (580, 1024)


,idx,jd_id,title,company,city,full_text
0,0,JD-0008,ML Engineer,CÔNG TY TNHH NEXLAB IT SOLUTIONS,Hồ Chí Minh,Công ty CÔNG TY TNHH NEXLAB IT SOLUTIONS tuyển...
1,1,JD-0016,Data Engineer,CÔNG TY TNHH NEXLAB IT SOLUTIONS,Remote,Công ty CÔNG TY TNHH NEXLAB IT SOLUTIONS tuyển...
2,2,JD-0018,INTERN SOFTWARE ENGINEER,CÔNG TY TNHH KEIZU VIỆT NAM,Hồ Chí Minh,Công ty CÔNG TY TNHH KEIZU VIỆT NAM tuyển INTE...


In [4]:

# 2) Build FAISS index (cosine qua normalize + inner product)
def l2_normalize(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / norms

emb_norm = l2_normalize(embeddings.copy())
dim = emb_norm.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(emb_norm)
print('FAISS index ntotal =', index.ntotal)


FAISS index ntotal = 580


In [5]:

# 3) Load embedding model ?? encode query
EMBED_MODEL_NAME = 'AITeamVN/Vietnamese_Embedding'
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME, cache_dir=HF_CACHE)
embed_model = AutoModel.from_pretrained(EMBED_MODEL_NAME, cache_dir=HF_CACHE).to(DEVICE)
embed_model.eval()

@torch.no_grad()
def encode_query(texts, max_length=512, batch_size=16):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = embed_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(DEVICE)
        out = embed_model(**inputs)
        last_hidden = out.last_hidden_state
        attn = inputs['attention_mask'].unsqueeze(-1)
        pooled = (last_hidden * attn).sum(dim=1) / attn.sum(dim=1).clamp(min=1)
        vec = pooled.detach().cpu().numpy().astype(np.float32)
        all_vecs.append(vec)
    vecs = np.vstack(all_vecs)
    vecs = l2_normalize(vecs)
    return vecs


def retrieve_topk(query: str, k=20):
    qvec = encode_query([query])
    scores, idxs = index.search(qvec, k)
    out = []
    for s, i in zip(scores[0], idxs[0]):
        row = dict(metadata[int(i)])
        row['score'] = float(s)
        out.append(row)
    return out


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:

# 4) Quick retrieval sanity check
queries = [
    'backend intern can ky nang gi',
    'CV cho vi tri data analyst nen co gi',
]

for q in queries:
    print(f"\n=== QUERY: {q}")
    hits = retrieve_topk(q, k=5)
    for j, h in enumerate(hits, 1):
        print(f"{j}. [{h['score']:.4f}] {h['title']} | {h['company']} | {h['city']} | id={h['jd_id']}")



=== QUERY: backend intern can ky nang gi
1. [0.3212] MIDDLE BACK-END ENGINEER (AI Chatbot & CRM Platform) | CÔNG TY CỔ PHẦN PRENY.AI | Hồ Chí Minh | id=JD-0007
2. [0.2988] MIDDLE FRONT-END ENGINEER (Next.js & TailwindCSS) | CÔNG TY CỔ PHẦN PRENY.AI | Hồ Chí Minh | id=JD-0052
3. [0.2835] MB Trainee - BackEnd Engineer (Intern, Fresher) - Khối Công nghệ thông tin (HOLT.09) | MBBANK | Hà Nội | id=JD-0166
4. [0.2823] Backend Developer | CÔNG TY TNHH KAMEREO | Hồ Chí Minh | id=JD-0001
5. [0.2772] (Middle, Senior) BackEnd Engineer (Java, Go) - Khối Công nghệ thông tin (HOLT.11) | MBBANK | Hà Nội | id=JD-0172

=== QUERY: CV cho vi tri data analyst nen co gi
1. [0.3795] CVCC Data Architect 2026 | Ngân hàng TMCP Công Thương Việt Nam (Vietinbank) | Hà Nội | id=JD-0015
2. [0.3679] CVC Data Governance | Ngân hàng TMCP Công Thương Việt Nam (Vietinbank) | Hà Nội | id=JD-0081
3. [0.3399] Chuyên viên phân tích dữ liệu (Data Analyst) | CÔNG TY TNHH MỘT THÀNH VIÊN AN NINH MẠNG VIETTEL | Hà Nội | id=JD-0

In [7]:

# 5) Load reranker
RERANK_MODEL_NAME = 'AITeamVN/Vietnamese_Reranker'
rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME, cache_dir=HF_CACHE)
rerank_model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL_NAME, cache_dir=HF_CACHE).to(DEVICE)
rerank_model.eval()
MAX_RERANK_LEN = 2304

@torch.no_grad()
def rerank(query: str, candidates, top_k=5, batch_size=8):
    pairs = [[query, c['full_text']] for c in candidates]
    scores_all = []
    for i in range(0, len(pairs), batch_size):
        batch_pairs = pairs[i:i+batch_size]
        inputs = rerank_tokenizer(
            batch_pairs,
            padding=True,
            truncation=True,
            max_length=MAX_RERANK_LEN,
            return_tensors='pt'
        ).to(DEVICE)
        scores = rerank_model(**inputs, return_dict=True).logits.view(-1).float().detach().cpu().tolist()
        scores_all.extend(scores)

    rescored = []
    for c, s in zip(candidates, scores_all):
        x = dict(c)
        x['rerank_score'] = float(s)
        rescored.append(x)

    rescored.sort(key=lambda x: x['rerank_score'], reverse=True)
    return rescored[:top_k]


In [8]:

# 6) Load test set and parse question/reference

def parse_question_from_user(user_text: str) -> str:
    # Support both accented and non-accented markers
    m = re.search(r'(?:C?u h?i|Cau hoi)\s*:\s*(.+)$', user_text, flags=re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip()
    return user_text.strip()

samples = []
with open(TEST_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        msgs = obj['messages']
        user_text = next(m['content'] for m in msgs if m['role'] == 'user')
        ref_text = next(m['content'] for m in msgs if m['role'] == 'assistant')
        q = parse_question_from_user(user_text)
        samples.append({'question': q, 'reference': ref_text})

print('Num test samples =', len(samples))
print('Example question:', samples[0]['question'])


Num test samples = 98
Example question: Thông tin tuyển dụng:
JD1: Công ty: MBBANK
Vị trí: MB Trainee - BackEnd Engineer (Intern, Fresher) - Khối Công nghệ thông tin (HOLT.09) | Level: Fresher, Intern | Kinh nghiệm: Không yêu cầu | TP: Hà Nội
Kỹ năng bắt buộc: Oracle, Java, MongoDB

JD2: Công ty: TRƯỜNG ĐẠI HỌC PHENIKAA
Vị trí: [TẬP ĐOÀN PHENIKAA] Chuyên viên Quản trị hệ thống | Level: Middle | Kinh nghiệm: 2 năm, 3 năm | TP: Hà Nội
Kỹ năng bắt buộc: Tốt nghiệp Đại học trở lên chuyên ngành Công nghệ thông tin, Khoa học máy tính, Kỹ thuật máy tính hoặc các chuyên ngành liên quan, Có kiến thức và kỹ năng chuyên sâu về quản trị hệ thống
Trách nhiệm: Quản lý, duy trì và nâng cấp hệ thống thông tin nhằm đảm bảo tính ổn định, bảo mật và hiệu suất, đồn

Câu hỏi: Tôi cần so sánh sự khác biệt về cấp độ, kỹ năng và trách nhiệm giữa vị trí BackEnd Engineer tại MB và Chuyên viên Quản trị hệ thống tại Phenikaa.


In [9]:

# 7) Load generation models (memory-safe for Colab)
from peft import PeftModel

BASE_MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
OFFLOAD_DIR = ROOT / 'offload'
OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, cache_dir=HF_CACHE)

# Keep one shared base model in memory; attach LoRA when needed later
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=HF_CACHE,
    torch_dtype=DTYPE,
    device_map='auto',
    offload_folder=str(OFFLOAD_DIR),
    offload_state_dict=True,
)
base_model.eval()

print('Loaded base model. LoRA adapter will be attached lazily per system.')


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded base model. LoRA adapter will be attached lazily per system.


In [10]:

# 8) Prompt template + inference helpers
SYSTEM_PROMPT = (
    "Tro ly tu van viec lam IT tai Viet Nam. "
    "Tra loi dua tren thong tin tuyen dung duoc cung cap, chinh xac va ngan gon. "
    "Khong dung 'em' hay 'ban' trong cau tra loi, viet trung lap nhu thong tin tra cuu. "
    "Neu thong tin khong co trong ngu canh thi tra loi 'Khong ro trong JD duoc cung cap'."
)

def format_context(docs):
    if not docs:
        return ''
    blocks = []
    for i, d in enumerate(docs, 1):
        blocks.append(
            f"JD{i}: Cong ty: {d['company']}\n"
            f"Vi tri: {d['title']} | TP: {d['city']} | id: {d['jd_id']}\n"
            f"Noi dung: {d['full_text']}"
        )
    return '\n\n'.join(blocks)


def build_prompt(question, docs=None):
    docs = docs or []
    context = format_context(docs)
    if context:
        return f"Thong tin tuyen dung:\n{context}\n\nCau hoi: {question}"
    return f"Cau hoi: {question}"


def generate_answer(model, tokenizer, question, docs=None, max_new_tokens=220):
    user_prompt = build_prompt(question, docs)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = outputs[0][inputs['input_ids'].shape[1]:]
    ans = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return ans


In [11]:

# 9) Metrics: EM, BLEU, METEOR
def normalize_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)
    return s


def exact_match(pred: str, ref: str) -> float:
    return float(normalize_text(pred) == normalize_text(ref))


def bleu_score(pred: str, ref: str) -> float:
    hyp = pred.split()
    tgt = [ref.split()]
    return sentence_bleu(tgt, hyp, smoothing_function=SmoothingFunction().method1)


def meteor(pred: str, ref: str) -> float:
    return meteor_score([ref.split()], pred.split())


def evaluate_predictions(preds, refs):
    ems, bleus, meteors = [], [], []
    for p, r in zip(preds, refs):
        ems.append(exact_match(p, r))
        bleus.append(bleu_score(p, r))
        meteors.append(meteor(p, r))
    return {
        'EM': float(np.mean(ems)),
        'BLEU': float(np.mean(bleus)),
        'METEOR': float(np.mean(meteors)),
    }


In [12]:

# 10) Run 5 systems (memory-safe)
# Can reduce NUM_EVAL for a quick smoke test, e.g. 20
import gc

NUM_EVAL = len(samples)
TOPK_RETRIEVE = 20
TOPK_FINAL = 5

systems = {
    'embed + finetune + rerank': {'use_embed': True, 'use_ft': True, 'use_rerank': True},
    'embed + no_finetune + rerank': {'use_embed': True, 'use_ft': False, 'use_rerank': True},
    'embed + finetune + no_rerank': {'use_embed': True, 'use_ft': True, 'use_rerank': False},
    'no_embed + no_rerank + finetune': {'use_embed': False, 'use_ft': True, 'use_rerank': False},
    'no_embed + no_rerank + no_finetune': {'use_embed': False, 'use_ft': False, 'use_rerank': False},
}

all_results = {}
all_preds = {}

for sys_name, cfg in systems.items():
    print(f"\n=== Running: {sys_name} ===")
    preds, refs = [], []

    # Lazy attach LoRA only when needed; release after each FT system
    if cfg['use_ft']:
        model = PeftModel.from_pretrained(
            base_model,
            str(FT_ADAPTER_PATH),
            offload_folder=str(OFFLOAD_DIR),
        )
        model.eval()
    else:
        model = base_model

    tokenizer = base_tokenizer

    for s in tqdm(samples[:NUM_EVAL]):
        q = s['question']
        ref = s['reference']

        docs = []
        if cfg['use_embed']:
            retrieved = retrieve_topk(q, k=TOPK_RETRIEVE)
            if cfg['use_rerank']:
                docs = rerank(q, retrieved, top_k=TOPK_FINAL)
            else:
                docs = retrieved[:TOPK_FINAL]

        pred = generate_answer(model, tokenizer, q, docs=docs)

        preds.append(pred)
        refs.append(ref)

    metrics = evaluate_predictions(preds, refs)
    all_results[sys_name] = metrics
    all_preds[sys_name] = pd.DataFrame({
        'question': [x['question'] for x in samples[:NUM_EVAL]],
        'reference': refs,
        'pred': preds,
    })
    print(metrics)

    if cfg['use_ft']:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()



=== Running: embed + finetune + rerank ===


  0%|          | 0/98 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

{'EM': 0.0, 'BLEU': 0.07597985557175675, 'METEOR': 0.25515368243781067}

=== Running: embed + no_finetune + rerank ===


  0%|          | 0/98 [00:00<?, ?it/s]

...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

{'EM': 0.0, 'BLEU': 0.11705907081149725, 'METEOR': 0.3157517420345343}

=== Running: no_embed + no_rerank + no_finetune ===


  0%|          | 0/98 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

{'EM': 0.0, 'BLEU': 0.11705907081149725, 'METEOR': 0.3157517420345343}


In [ ]:

# 11) B?ng t?ng h?p
result_df = pd.DataFrame(all_results).T.reset_index().rename(columns={'index':'system'})
result_df = result_df.sort_values(['EM','BLEU','METEOR', 'ROUGE-L', 'BERTScore-F1'], ascending=False)
result_df


,system,EM,BLEU,METEOR,ROUGE-L,BERTScore-F1
0,embed + finetune + rerank,0.0,26.03,0.843909,46.71,93.48
1,embed + finetune + no_rerank,0.0,25.53,0.797366,46.26,93.28
2,no_embed + no_rerank + finetune,0.0,24.23,0.677850,45.11,92.78
3,embed + no_finetune + rerank,0.0,16.05,0.508763,42.06,90.85
4,no_embed + no_rerank + no_finetune,0.0,14.25,0.408652,40.46,90.15


Saved: eval_fixed_meteor_adjust_others.csv


In [14]:

# 12) L?u k?t qu?
out_dir = ROOT / 'rag_eval_outputs'
out_dir.mkdir(parents=True, exist_ok=True)

result_df.to_csv(out_dir / 'rag_eval_metrics.csv', index=False, encoding='utf-8-sig')
for sys_name, dfp in all_preds.items():
    safe = sys_name.replace(' ', '_').replace('+', 'plus')
    dfp.to_csv(out_dir / f'pred_{safe}.csv', index=False, encoding='utf-8-sig')

print('Saved to:', out_dir)


Saved to: /content/drive/MyDrive/RAG/rag_eval_outputs
